In [2]:
import pandas as pd
import random

# Five domains
categories = ["Water Supply", "Electricity", "Waste Management", "Roads", "Healthcare"]

# Example complaint samples for each category
complaint_samples = {
    "Water Supply": [
        "No water supply since morning in our area",
        "Tap water is muddy and smells bad",
        "Water leakage from main pipeline near the park",
        "Low water pressure in the evening hours",
        "Water tank cleaning not done for months"
    ],
    "Electricity": [
        "Power cut in my locality since last night",
        "Street lights are not working on our road",
        "Voltage fluctuations damaging appliances",
        "Electric pole fallen after rain",
        "Transformer burst causing power outage"
    ],
    "Waste Management": [
        "Garbage not collected from my area for 3 days",
        "Bins are overflowing near the market",
        "Improper disposal of plastic waste",
        "Animals scattering garbage on roads",
        "Need more dustbins in residential area"
    ],
    "Roads": [
        "Potholes on main road causing accidents",
        "Road not repaired after sewer work",
        "Broken divider leading to traffic jam",
        "Street under construction for months",
        "Need new speed breakers near school"
    ],
    "Healthcare": [
        "No doctor available in the government hospital",
        "Medicines not available in dispensary",
        "Ambulance delayed during emergency",
        "Poor hygiene in the health centre",
        "Need more staff in primary health centre"
    ]
}

# Generate synthetic dataset (100 entries per category)
data = []
for category in categories:
    for _ in range(100):
        text = random.choice(complaint_samples[category])
        data.append({
            "complaint_text": text,
            "category": category
        })

# Convert to DataFrame
df = pd.DataFrame(data)

# Shuffle rows for randomness
df = df.sample(frac=1).reset_index(drop=True)

# Save to CSV
df.to_csv("complaints_dataset.csv", index=False)


In [5]:
pip install textblob

   ---------------------------------------- 0.0/624.3 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/624.3 kB ? eta -:--:--
   ---------------------------------------- 624.3/624.3 kB 2.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from textblob import TextBlob
import joblib

# Load dataset
df = pd.read_csv("complaints_dataset.csv")

# Create pipeline (tokenization + TF-IDF + classifier)
model = Pipeline([
    ('vect', CountVectorizer(max_features=500, lowercase=True, stop_words='english')),
    ('tfidf', TfidfTransformer()),
    ('clf', RandomForestClassifier(n_estimators=120, random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['complaint_text'], df['category'], test_size=0.25, random_state=42
)

# Train model
model.fit(X_train, y_train)

# Evaluate
pred = model.predict(X_test)
print("\n✅ Accuracy:", round(accuracy_score(y_test, pred), 3))
print("\n📊 Classification Report:\n", classification_report(y_test, pred))

# Save trained model
joblib.dump(model, "complaint_classifier.pkl")
print("\n💾 Model saved successfully as 'complaint_classifier.pkl'!")

# Function to compute priority score
def get_priority_score(text):
    polarity = TextBlob(text).sentiment.polarity  # -1 to +1
    return round((1 - polarity) * 5, 2)  # higher score → more negative → higher priority

# Test with sample complaint
sample = "No electricity in my area since morning"
pred_class = model.predict([sample])[0]
priority = get_priority_score(sample)


✅ Accuracy: 1.0

📊 Classification Report:
                   precision    recall  f1-score   support

     Electricity       1.00      1.00      1.00        23
      Healthcare       1.00      1.00      1.00        23
           Roads       1.00      1.00      1.00        25
Waste Management       1.00      1.00      1.00        30
    Water Supply       1.00      1.00      1.00        24

        accuracy                           1.00       125
       macro avg       1.00      1.00      1.00       125
    weighted avg       1.00      1.00      1.00       125


💾 Model saved successfully as 'complaint_classifier.pkl'!


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

# Example pipeline with GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', RandomForestClassifier())
])

# Define grid for hyperparameter tuning
param_grid = {
    'tfidf__max_df': [0.8, 0.9, 1.0],
    'tfidf__min_df': [1, 2, 3],
    'clf__n_estimators': [50, 100, 150],
    'clf__max_depth': [None, 10, 20]
}

# Perform grid search
grid = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=-1, verbose=2)
grid.fit(X_train, y_train)

print("✅ Best parameters found:", grid.best_params_)
print("🔍 Best cross-validation score:", grid.best_score_)

y_pred = grid.predict(X_test)
print("\nFinal Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Fitting 3 folds for each of 81 candidates, totalling 243 fits
✅ Best parameters found: {'clf__max_depth': None, 'clf__n_estimators': 50, 'tfidf__max_df': 0.8, 'tfidf__min_df': 1}
🔍 Best cross-validation score: 1.0

Final Accuracy: 1.0

Classification Report:
                   precision    recall  f1-score   support

     Electricity       1.00      1.00      1.00        23
      Healthcare       1.00      1.00      1.00        23
           Roads       1.00      1.00      1.00        25
Waste Management       1.00      1.00      1.00        30
    Water Supply       1.00      1.00      1.00        24

        accuracy                           1.00       125
       macro avg       1.00      1.00      1.00       125
    weighted avg       1.00      1.00      1.00       125



In [8]:
import pandas as pd
import random

domains = ['Water Supply', 'Electricity', 'Waste Management', 'Roads', 'Healthcare']

base_complaints = {
    'Water Supply': [
        "No water coming since morning", "Water leakage near my house", 
        "Dirty water from tap", "Low water pressure", "Water supply is irregular"
    ],
    'Electricity': [
        "Power cut for several hours", "Street lights not working", 
        "Electricity fluctuation damages appliances", "Transformer sparks", "No electricity in area"
    ],
    'Waste Management': [
        "Garbage not collected", "Dustbin overflowing", "Bad smell from waste", 
        "No proper waste disposal", "Mosquitoes due to garbage"
    ],
    'Roads': [
        "Road full of potholes", "Broken road causing traffic", 
        "Need road repair", "New road construction delayed", "Uneven roads in colony"
    ],
    'Healthcare': [
        "Doctor not available in clinic", "Medicine stock finished", 
        "Ambulance not responding", "Long waiting time in hospital", "Need more staff in healthcare center"
    ]
}

def add_variation(text):
    noise = [
        "please check", "kindly fix", "urgent issue", "very serious", 
        "not resolved yet", "happening since a week", "citizens facing problems", ""
    ]
    typo = random.choice(["", "plz", "urgnt", "help", "issue"])
    return f"{text} {random.choice(noise)} {typo}"

data = []
for domain, texts in base_complaints.items():
    for _ in range(80):  # 80 per category → 400 total
        text = random.choice(texts)
        # Add some cross-domain noise
        if random.random() < 0.15:
            text += " " + random.choice(random.choice(list(base_complaints.values())))
        data.append({
            'complaint': add_variation(text),
            'category': domain
        })

df = pd.DataFrame(data)
df.to_csv("realistic_complaints.csv", index=False)
print(df.head())

                                      complaint      category
0  Water supply is irregular please check issue  Water Supply
1                   Dirty water from tap  urgnt  Water Supply
2             Low water pressure kindly fix plz  Water Supply
3   Water supply is irregular very serious help  Water Supply
4     No water coming since morning kindly fix   Water Supply


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv("realistic_complaints.csv")

X_train, X_test, y_train, y_test = train_test_split(df['complaint'], df['category'], test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), stop_words='english', max_features=2000)),
    ('clf', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.95

Classification Report:
                   precision    recall  f1-score   support

     Electricity       1.00      1.00      1.00        17
      Healthcare       0.94      0.94      0.94        18
           Roads       0.85      1.00      0.92        11
Waste Management       1.00      1.00      1.00        12
    Water Supply       0.95      0.86      0.90        22

        accuracy                           0.95        80
       macro avg       0.95      0.96      0.95        80
    weighted avg       0.95      0.95      0.95        80



In [10]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("realistic_complaints.csv")

X_train, X_test, y_train, y_test = train_test_split(
    df['complaint'], df['category'], test_size=0.2, random_state=42
)

# Store model performance
results = []

# ----------------------------------
# 1️⃣ Logistic Regression
# ----------------------------------
logreg_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=2000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000))
])

logreg_params = {
    'clf__C': [0.1, 1, 5, 10],
    'clf__penalty': ['l2'],
    'tfidf__max_df': [0.8, 1.0],
}

grid_logreg = GridSearchCV(logreg_pipeline, logreg_params, cv=3, n_jobs=-1)
grid_logreg.fit(X_train, y_train)
y_pred = grid_logreg.predict(X_test)
acc = accuracy_score(y_test, y_pred)
results.append(("Logistic Regression", acc, grid_logreg.best_params_))
print("\n🔹 Logistic Regression Accuracy:", acc)

# ----------------------------------
# 2️⃣ Random Forest
# ----------------------------------
rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=2000)),
    ('clf', RandomForestClassifier())
])

rf_params = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5],
}

grid_rf = GridSearchCV(rf_pipeline, rf_params, cv=3, n_jobs=-1)
grid_rf.fit(X_train, y_train)
y_pred = grid_rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
results.append(("Random Forest", acc, grid_rf.best_params_))
print("🔹 Random Forest Accuracy:", acc)

# ----------------------------------
# 3️⃣ Support Vector Machine
# ----------------------------------
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=2000)),
    ('clf', LinearSVC())
])

svm_params = {
    'clf__C': [0.1, 1, 5, 10],
    'tfidf__ngram_range': [(1,1), (1,2)]
}

grid_svm = GridSearchCV(svm_pipeline, svm_params, cv=3, n_jobs=-1)
grid_svm.fit(X_train, y_train)
y_pred = grid_svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)
results.append(("SVM", acc, grid_svm.best_params_))
print("🔹 SVM Accuracy:", acc)

# ----------------------------------
# 4️⃣ Naive Bayes
# ----------------------------------
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=2000)),
    ('clf', MultinomialNB())
])

nb_params = {
    'clf__alpha': [0.1, 0.5, 1.0],
    'tfidf__ngram_range': [(1,1), (1,2)]
}

grid_nb = GridSearchCV(nb_pipeline, nb_params, cv=3, n_jobs=-1)
grid_nb.fit(X_train, y_train)
y_pred = grid_nb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
results.append(("Naive Bayes", acc, grid_nb.best_params_))
print("🔹 Naive Bayes Accuracy:", acc)

# ----------------------------------
# 🏆 Compare and Save the Best Model
# ----------------------------------
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Best Params"])
print("\n📊 Model Comparison:\n", results_df)

best_model_row = results_df.loc[results_df['Accuracy'].idxmax()]
best_model_name = best_model_row['Model']
print(f"\n✅ Best Model: {best_model_name} (Accuracy: {best_model_row['Accuracy']:.2f})")

# Save the best model
if best_model_name == "Logistic Regression":
    best_model = grid_logreg
elif best_model_name == "Random Forest":
    best_model = grid_rf
elif best_model_name == "SVM":
    best_model = grid_svm
else:
    best_model = grid_nb

joblib.dump(best_model, "best_complaint_classifier.pkl")
print("\n💾 Best model saved as 'best_complaint_classifier.pkl'")



🔹 Logistic Regression Accuracy: 0.95
🔹 Random Forest Accuracy: 0.95
🔹 SVM Accuracy: 0.9625
🔹 Naive Bayes Accuracy: 0.9375

📊 Model Comparison:
                  Model  Accuracy  \
0  Logistic Regression    0.9500   
1        Random Forest    0.9500   
2                  SVM    0.9625   
3          Naive Bayes    0.9375   

                                         Best Params  
0  {'clf__C': 10, 'clf__penalty': 'l2', 'tfidf__m...  
1  {'clf__max_depth': 20, 'clf__min_samples_split...  
2       {'clf__C': 10, 'tfidf__ngram_range': (1, 2)}  
3  {'clf__alpha': 1.0, 'tfidf__ngram_range': (1, 1)}  

✅ Best Model: SVM (Accuracy: 0.96)

💾 Best model saved as 'best_complaint_classifier.pkl'


In [11]:
from sklearn.metrics import classification_report

print("\n🔍 Generating full classification report for the best model...")

y_best_pred = best_model.predict(X_test)
print("\n📋 Classification Report for Best Model:\n")
print(classification_report(y_test, y_best_pred))


🔍 Generating full classification report for the best model...

📋 Classification Report for Best Model:

                  precision    recall  f1-score   support

     Electricity       1.00      1.00      1.00        17
      Healthcare       0.95      1.00      0.97        18
           Roads       0.85      1.00      0.92        11
Waste Management       1.00      1.00      1.00        12
    Water Supply       1.00      0.86      0.93        22

        accuracy                           0.96        80
       macro avg       0.96      0.97      0.96        80
    weighted avg       0.97      0.96      0.96        80



In [13]:
pip install nltk


Note: you may need to restart the kernel to use updated packages.


In [14]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation/numbers
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)


In [24]:
import pandas as pd
import re
import nltk
import joblib
from textblob import TextBlob
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings("ignore")

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')

# ---------------------------
# Step 1️⃣ - Text Cleaning
# ---------------------------
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r'[^a-zA-Z ]', '', text.lower())  # keep only letters
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

# Load dataset
df = pd.read_csv("realistic_complaints.csv")
df['cleaned_complaint'] = df['complaint'].apply(clean_text)

X = df['cleaned_complaint']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# Step 2️⃣ - Try Multiple Models
# ---------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=2),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42),
    "SVM": LinearSVC(C=1),
    "Naive Bayes": MultinomialNB(alpha=0.3)
}

results = []

for name, model in models.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=200, ngram_range=(1,2))),
        ('clf', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n🔹 {name} Accuracy: {acc:.3f}")
    print(classification_report(y_test, y_pred))
    results.append((name, acc, pipeline))

# ---------------------------
# Step 3️⃣ - Select Best Model
# ---------------------------
best_model_name, best_acc, best_pipeline = max(results, key=lambda x: x[1])
print(f"\n✅ Best Model: {best_model_name} with Accuracy: {best_acc:.3f}")

joblib.dump(best_pipeline, "optimized_complaint_classifier.pkl")
print("💾 Saved as 'optimized_complaint_classifier.pkl'")

# ---------------------------
# Step 4️⃣ - Priority Scoring Function
# ---------------------------
def get_priority_score(text):
    polarity = TextBlob(text).sentiment.polarity
    score = round((1 - polarity) * 5, 2)
    if score >= 8:
        level = "High"
    elif score >= 5:
        level = "Medium"
    else:
        level = "Low"
    return level, score

# ---------------------------
# Step 5️⃣ - Test Prediction
# ---------------------------
def predict_complaint_priority(text):
    cleaned = clean_text(text)
    model = joblib.load("optimized_complaint_classifier.pkl")
    category = model.predict([cleaned])[0]
    priority_label, priority_score = get_priority_score(text)
    return {
        "Complaint": text,
        "Predicted Category": category,
        "Priority Level": priority_label,
        "Priority Score": priority_score
    }

# Test example
sample = "The water pipeline has been leaking for three days, and nobody from the maintenance department has come yet."
result = predict_complaint_priority(sample)
print("\n🚨 Sample Prediction:\n", result)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mohit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mohit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



🔹 Logistic Regression Accuracy: 0.950
                  precision    recall  f1-score   support

     Electricity       1.00      1.00      1.00        17
      Healthcare       0.94      0.94      0.94        18
           Roads       0.85      1.00      0.92        11
Waste Management       1.00      1.00      1.00        12
    Water Supply       0.95      0.86      0.90        22

        accuracy                           0.95        80
       macro avg       0.95      0.96      0.95        80
    weighted avg       0.95      0.95      0.95        80


🔹 Random Forest Accuracy: 0.963
                  precision    recall  f1-score   support

     Electricity       1.00      1.00      1.00        17
      Healthcare       0.94      0.94      0.94        18
           Roads       0.92      1.00      0.96        11
Waste Management       1.00      1.00      1.00        12
    Water Supply       0.95      0.91      0.93        22

        accuracy                           0.96      

In [25]:
# ---------------------------
# Step 6️⃣ - Print all Hyperparameters of the Best Model
# ---------------------------
print("\n🔧 All Hyperparameters of the Best Model:\n")

# Extract the vectorizer and classifier
tfidf = best_pipeline.named_steps['tfidf']
clf = best_pipeline.named_steps['clf']

# Print TF-IDF parameters
print("🧩 TF-IDF Vectorizer Parameters:")
for param, value in tfidf.get_params().items():
    print(f"  {param}: {value}")

# Print classifier parameters
print("\n🌲 Random Forest Classifier Parameters:")
for param, value in clf.get_params().items():
    print(f"  {param}: {value}")


🔧 All Hyperparameters of the Best Model:

🧩 TF-IDF Vectorizer Parameters:
  analyzer: word
  binary: False
  decode_error: strict
  dtype: <class 'numpy.float64'>
  encoding: utf-8
  input: content
  lowercase: True
  max_df: 1.0
  max_features: 200
  min_df: 1
  ngram_range: (1, 2)
  norm: l2
  preprocessor: None
  smooth_idf: True
  stop_words: None
  strip_accents: None
  sublinear_tf: False
  token_pattern: (?u)\b\w\w+\b
  tokenizer: None
  use_idf: True
  vocabulary: None

🌲 Random Forest Classifier Parameters:
  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 15
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 1
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 150
  n_jobs: None
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False
